# Lab 13 solution, notebook 1: a question one search answers, and a question that needs one more sentence

The worked answers to the two challenge tasks of `13a_meet_agentic_rag.ipynb`. The setup repeats what the tasks need
from the notebook (the notes, the search, the tool, the system messages and `run_loop`), so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
import json
import numpy as np
from openai import OpenAI, omit

client = OpenAI(timeout=60)
EMBEDDING_MODEL = settings["EMBEDDING_MODEL"]

# The notes, embedded in one request: row i of D is the vector of names[i]
NOTES_FOLDER = ROOT / "13_Agentic_RAG" / "my_notes"
NOTES = {path.name: path.read_text().strip() for path in sorted(NOTES_FOLDER.glob("*.md"))}
names = list(NOTES)
response = client.embeddings.create(model=EMBEDDING_MODEL, input=list(NOTES.values()))
D = np.array([item.embedding for item in response.data])


def search(query, k=2):
    """Return the k notes nearest to the query, one per line: [file name] text."""
    q = np.array(client.embeddings.create(model=EMBEDDING_MODEL, input=query).data[0].embedding)
    best = np.argsort(-(D @ q))[:k]
    return "\n".join(f"[{names[i]}] {NOTES[names[i]]}" for i in best)

In [3]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search my notes with a short query of a few words. Returns the two nearest notes.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

SYSTEM = "Answer only from my notes. Use the search tool to find them. If the notes do not say, say you do not know."
SYSTEM_AGAIN = SYSTEM + " Search again while the question is still open."

In [4]:
def run_loop(question, system, max_turns=5):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question},
    ]
    queries = []

    for turn in range(1, max_turns + 1):
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=[search_tool], reasoning_effort=EFFORT or omit
        )
        reply = response.choices[0].message
        messages.append(reply)

        # The model answered: return its text and every query it wrote
        if not reply.tool_calls:
            return reply.content, queries

        for call in reply.tool_calls:
            query = json.loads(call.function.arguments)["query"]
            result = search(query)
            print(f"turn {turn}: search({query!r})\n    " + result.replace("\n", "\n    "))
            queries.append(query)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

    # The turn cap stopped the loop: no answer
    return None, queries

## Task 1: A question the first search answers

The GP's address is in `okafor.md`, a note whose words ("Dr Okafor's surgery") are close to the question's.

In [5]:
answer, queries = run_loop("Where is my GP's surgery?", SYSTEM_AGAIN)
print()
print("searches:", len(queries))
print("answer:", answer)

turn 1: search('GP surgery address location')
    [okafor.md] Dr Okafor's surgery: 3 Mill Lane. Open 8:00 to 18:00. Book online.
    [vet.md] Biscuit's vet: Park Road Animal Clinic, 21 Park Road. Jabs due in May.


turn 2: search('Dr Okafor surgery address')
    [okafor.md] Dr Okafor's surgery: 3 Mill Lane. Open 8:00 to 18:00. Book online.
    [health.md] My dentist is Mia Rana. My GP is Dr Okafor. Check-ups with both in March.



searches: 2
answer: Your GP, Dr Okafor’s surgery, is at **3 Mill Lane**.


The first search, *GP surgery address location*, returned `okafor.md` with the address. This run then searched once more, *Dr Okafor surgery address*, with the name from the notes, and answered: *“Your GP, Dr Okafor’s surgery, is at **3 Mill Lane**.”* Two searches here; most runs stop after the first. Either way the answer was in the first result, and no second search was needed to find it.

## Task 2: A question where the added sentence is not enough

First with `SYSTEM_AGAIN`, as in the notebook. Read the second query.

In [6]:
answer, queries = run_loop("Which day is my dentist closed?", SYSTEM_AGAIN)
print()
print("answer:", answer)

turn 1: search('dentist closed day')
    [okafor.md] Dr Okafor's surgery: 3 Mill Lane. Open 8:00 to 18:00. Book online.
    [health.md] My dentist is Mia Rana. My GP is Dr Okafor. Check-ups with both in March.


turn 2: search('dentist opening hours closed')
    [okafor.md] Dr Okafor's surgery: 3 Mill Lane. Open 8:00 to 18:00. Book online.
    [health.md] My dentist is Mia Rana. My GP is Dr Okafor. Check-ups with both in March.



answer: I don’t know—the notes don’t say which day your dentist is closed.


Two searches, *dentist closed day* and *dentist opening hours closed*, both from the question's words, and both returned `okafor.md` and `health.md`. The name Mia Rana reached the model in `health.md` on turn 1, and the second query did not use it, so `rana.md` ("Closed on Fridays") never came back. The answer: *“I don’t know—the notes don’t say which day your dentist is closed.”*

The fix is one more sentence that says what to search for: a name the notes give for the person or place the question
asks about.

In [7]:
SYSTEM_NAMES = SYSTEM_AGAIN + " When a note names the person or place you need, search for that name."

answer, queries = run_loop("Which day is my dentist closed?", SYSTEM_NAMES)
print()
print("answer:", answer)

turn 1: search('dentist closed day')
    [okafor.md] Dr Okafor's surgery: 3 Mill Lane. Open 8:00 to 18:00. Book online.
    [health.md] My dentist is Mia Rana. My GP is Dr Okafor. Check-ups with both in March.


turn 2: search('Mia Rana dentist closed')
    [health.md] My dentist is Mia Rana. My GP is Dr Okafor. Check-ups with both in March.
    [rana.md] Mia Rana: 12 Garden Street, second floor. Ring the top bell. Closed on Fridays.



answer: Your dentist, Mia Rana, is closed on Fridays.


Now turn 2 searched *Mia Rana dentist closed*, `rana.md` came back, and the answer is: *“Your dentist, Mia Rana, is closed on Fridays.”*

The new sentence must not break the question that already worked:

In [8]:
answer, queries = run_loop("What is the address of my dentist?", SYSTEM_NAMES)
print()
print("answer:", answer)

turn 1: search('dentist address')
    [health.md] My dentist is Mia Rana. My GP is Dr Okafor. Check-ups with both in March.
    [okafor.md] Dr Okafor's surgery: 3 Mill Lane. Open 8:00 to 18:00. Book online.


turn 2: search('Mia Rana')
    [rana.md] Mia Rana: 12 Garden Street, second floor. Ring the top bell. Closed on Fridays.
    [health.md] My dentist is Mia Rana. My GP is Dr Okafor. Check-ups with both in March.



answer: Your dentist, Mia Rana, is at **12 Garden Street, second floor**. Ring the top bell.


Still right: turn 2 searched *Mia Rana*, the name alone, and the answer gives 12 Garden Street, second floor.

**What this shows.** A system message steers which query the model writes next. "Search again" was enough when the
model's own second query happened to use the name; "search for that name" makes the name the query. Test a new
sentence on the questions that already worked, as the last cell does.